In [2]:
!pip install transformers

In [5]:
from transformers import AutoTokenizer ,AutoModelForCausalLM
import torch
import json

In [6]:
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [8]:
with open("input_prompts.json", "r") as f:
    data = json.load(f)

In [9]:
def direct_prompt(q):
    return f"Q: {q}\nA:"

In [10]:
def instruction_prompt(q):
    return f"Explain the following in a simple way for students:\n{q}"

In [11]:
def few_shot_prompt(q):
    return f"""
Q: What is gravity?
A: Gravity is a force that pulls objects toward each other.

Q: {q}
A:
"""

In [12]:
def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
results = []

for item in data:
    q = item["question"]

    res = {
        "question": q,
        "direct": generate_text(direct_prompt(q)),
        "instruction": generate_text(instruction_prompt(q)),
        "few_shot": generate_text(few_shot_prompt(q))
    }

    results.append(res)

In [14]:
with open("generated_results.json", "w") as f:
    json.dump(results, f, indent=4)